# `00_non_bicycle_route_ways`: Cyclable ways outside the bicycle route network

## Introduction

### Purpose

This notebook derives `non_bicycle_route_ways`: all OSM ways that are cyclable but do not belong to any bicycle route relation. Together with `bicycle_route_m_ways_distinct`, this set forms the complete cyclable network of the Netherlands:

```
non_bicycle_route_ways + bicycle_route_m_ways_distinct = entire cyclable network
```

In the broader pipeline, this layer serves as **scoping context** rather than as an object of analysis. The UNECE attribute-completeness assessment is applied only to the designated bicycle route network (`bicycle_route_m_ways_distinct`); the non-route layer is produced so that the two sides of the cyclable network are constructed by the same operationalisation, and so that the relation-bound scope of the analytical claims is explicit.

### Inputs

- OpenStreetMap data (HeiGIT ohsome planet snapshot, 6 October 2025, 13:16:28).
- `bicycle_route_m_ways_distinct`: distinct member ways of all bicycle route relations, clipped to the Netherlands (181,318 ways; 38,570.854 km).
- Dutch administrative area boundary dataset (`nl_admin_area_gdf`).

### Outputs

- `non_bicycle_route_ways`: cyclable OSM ways that are *not* members of any bicycle route relation, clipped to the Netherlands administrative boundary.

### Key steps

The derivation proceeds in two steps. First, the full universe of cyclable ways in the Netherlands (`all_cyclable_ways`) is identified using an inclusion-only filter based on explicit access tags (`bicycle=*` and its directional variants) and cycling infrastructure tags (`highway=cycleway`, `cycleway=lane/track/...`). The query is run against the OSM snapshot, then clipped to the Netherlands administrative boundary. Second, ways already captured in `bicycle_route_m_ways_distinct` are removed via an anti-join on `way_m_id`, leaving the set of cyclable ways that are unaffiliated with any designated bicycle route relation.

### Dependencies on prior notebooks

- `00_bicycle_route_relations.ipynb` provides `bicycle_route_m_ways_distinct` (for the anti-join), the OSM data path `osm_data_path`, and the helper `clip_osm_to_nl_admin_area`.
- `01_boundaries.ipynb` provides `nl_admin_area_gdf` used to clip ways to the Netherlands administrative boundary.

### Table of contents

1. [Environment setup](#1-environment-setup)
2. [Identify all cyclable ways](#2-identify-all-cyclable-ways)
   - 2.1 [What counts as cyclable](#21-what-counts-as-cyclable)
   - 2.2 [Build the cyclable-way table](#22-build-the-cyclable-way-table)
   - 2.3 [Clip to the Netherlands boundary](#23-clip-to-the-netherlands-boundary)
   - 2.4 [Limitations](#24-limitations)
3. [Exclude ways already part of a bicycle route network](#3-exclude-ways-already-part-of-a-bicycle-route-network)
4. [Result: `non_bicycle_route_ways`](#4-result-non_bicycle_route_ways)

---

## 1. Environment setup

### Libraries and extensions

In [4]:
from IPython.utils import io

### Loading shared variables

Execute the prior notebooks to bring `osm_data_path`, `bicycle_route_m_ways_distinct`, `clip_osm_to_nl_admin_area`, and `nl_admin_area_gdf` into the current session.

In [2]:
with io.capture_output() as captured:
    %run /home/vbo226/00_bicycle_route_relations.ipynb # osm_data_path and bicycle_route_m_ways_distinct
    %run /home/vbo226/01_boundaries.ipynb # nl_admin_area_gdf

---

## 1. Identify all cyclable ways
 
The first step constructs the full universe of cyclable ways in the Netherlands (`all_cyclable_ways`) without reference to route relations. This intermediate table is the foundation for the exclusion step that follows

### 2.1 What counts as cyclable

Determining whether a way is cyclable is not straightforward in OSM. Cyclability can be signalled in three distinct ways: through explicit access tags (e.g. `bicycle=yes/designated/permissive`), through the presence of cycling infrastructure (e.g. `highway=cycleway`, `cycleway=lane`), or through default access rules that apply when no tags are present.

In an ideal world, cycling legality would always be captured explicitly in `bicycle=*`, but the absence of `bicycle=*` is not in itself a problem or an oversight. It reflects how OSM tagging conventions handle redundancy, and these conventions differ by context. For `highway=cycleway`, adding `bicycle=designated` is explicitly discouraged: the highway type already defines the way as dedicated cycling infrastructure, and the access tag adds nothing. For carriageways with `cycleway=lane`, there is no equivalent convention against adding `bicycle=yes`, but in practice many mappers omit it: the presence of a cycle lane implies that cycling is permitted, and adding an explicit access tag can feel redundant. This is an understandable and common pattern, even if not a formally prescribed one. Where tagging becomes genuinely inconsistent is at the margins: some mappers do add access tags for clarity, others rely entirely on infrastructure tags, and others tag nothing at all, leaving cycling legality to be inferred from national [default access rules](https://wiki.openstreetmap.org/wiki/OSM_tags_for_routing/Access_restrictions#Netherlands). For the Netherlands, the default is that cycling is permitted on most roads unless explicitly prohibited, but this analysis does not attempt to resolve default access. Ways where cycling is only inferable from national defaults are excluded.

This dual reality, explicit access tags on some ways, infrastructure tags implying permission on others, directly shapes how the OSM access tag hierarchy is applied here. OSM encodes access through a hierarchy: `access=*` sets the broadest condition, `vehicle=*` narrows it to vehicles, and `bicycle=*` narrows it further to cycling specifically, with more specific tags overriding more general ones. In this hierarchy, `bicycle=*` is the tag that directly and unambiguously expresses whether cycling is permitted, and it is the only access tag evaluated here as a standalone criterion. General tags such as `access=yes` or `vehicle=no` are not used: using them as positive criteria would incorporate ways where cycling permission was never explicitly considered by the mapper, and using them as exclusion criteria would risk dropping genuinely cyclable ways. During the explicitness analysis, ways were observed carrying `vehicle=no` without any `bicycle=*` tag yet belonging to a bicycle route relation, suggesting that `vehicle=no` does not reliably reflect cycling access.

Both way-level `bicycle=*` and its directional variants (`bicycle:forward=*`, `bicycle:backward=*`) are evaluated, for the inclusion values `yes`, `designated`, `permissive`, and `discouraged`. Treating directional variants as equivalent to way-level tags is defensible because they serve the same purpose: they establish that the way participates in the cycling network in some form. `bicycle:forward=yes` does not say cycling is permitted in both directions, it says one direction is explicitly cyclable. `cycleway:right=lane` does not say infrastructure exists on both sides, it says infrastructure exists on one side. Both are partial signals rather than whole-way statements, and both are sufficient under the inclusion criterion adopted here: if cycling is permitted or infrastructure is present in at least one direction or on at least one side, the way is cyclable. From that perspective, `bicycle:forward/backward=*` and `cycleway:right/left=*` are logically equivalent signals and are treated as such.

Conditional tags (`access:conditional=*`, `vehicle:conditional=*`, `bicycle:conditional=*`) are not evaluated. They express access that applies only under specific conditions such as time of day or day of week, and general conditional tags interact with `bicycle:conditional=*` through the same access hierarchy. A `vehicle:conditional=no` at certain hours would propagate to cycling unless overridden by a more specific `bicycle:conditional=*`, making their joint resolution non-trivial. A way carrying only `bicycle:conditional=yes @ (Mo-Fr 07:00-09:00)` and no other inclusion tag is therefore excluded.

On the infrastructure side, the relevant signal is the presence of a cycling-specific tag that unambiguously describes a physical facility: `highway=cycleway`, `cycleway=lane`, `cycleway=track`, and similar values. Crucially, not all `cycleway=*` values signal the presence of cycling infrastructure on the way itself. `cycleway=no` indicates that no cycling facility is present, and `cycleway=separate` indicates that cycling infrastructure exists but is mapped as a separate OSM way rather than as an attribute of the carriageway. Neither implies that the way itself is cyclable, and neither is an inclusion criterion.

**The approach uses an inclusion-only filter:** a way is classified as cyclable if at least one of the tags listed below is present. Relying on access tags alone would produce a severe undercount precisely because of the common pattern described above, `bicycle=*` absent, infrastructure tag present. Infrastructure tags are therefore treated as equivalent signals of cyclability to explicit access tags, reflecting the dual tagging reality of OSM.

**Inclusion criteria.** A way is included if at least one of the following applies:

| Tag | Signal type | Notes |
|---|---|---|
| `bicycle = yes` (or `bicycle:forward/backward = yes`) | Explicit permission | |
| `bicycle = designated` (or directional variants) | Explicit permission | |
| `bicycle = permissive` (or directional variants) | Explicit permission | Allowed at landowner's discretion |
| `bicycle = discouraged` (or directional variants) | Explicit permission | Not recommended but not prohibited |
| `bicycle = optional_sidepath` (or directional variants) | Explicit permission | Parallel facility exists but not compulsory; carriageway remains cyclable |
| `highway = cycleway` | Infrastructure | Dedicated cycling way |
| `bicycle_road = yes` | Infrastructure / legal | Cycling street; cyclists have legal priority |
| `cyclestreet = yes` | Infrastructure / legal | Equivalent in Dutch context (*fietsstraat*) |
| `cycleway = lane` (or `cycleway:both/right/left = lane`) | Infrastructure | Unprotected lane on carriageway |
| `cycleway = shared_lane` (or directional variants) | Infrastructure | Shared lane marking |
| `cycleway = share_busway` (or directional variants) | Infrastructure | Shared with buses |
| `cycleway = track` (or directional variants) | Infrastructure | Protected track alongside carriageway |

**Exclusion as the absence of inclusion.** Rather than maintaining an explicit exclusion list applied as a hard override, ways are excluded simply by failing to satisfy any inclusion criterion. This is a deliberate choice that avoids a specific class of tagging ambiguity that is common in OSM. To understand why, it helps to consider what conflicting tags actually look like in practice. A real example from the OSM extract is a way tagged `{highway=cycleway, bicycle=no, moped=designated, mofa=no, foot=no}`. Here `bicycle=no` and `highway=cycleway` appear on the same way. One explanation is not that cycling is genuinely prohibited on a dedicated cycleway, but that the way expresses a more specific restriction, for example, that a particular type of bicycle or e-bike is excluded while the way remains a cycleway, and the `bicycle=no` tag is either a data quality error or a shorthand that collapses more nuanced access into a blunt way-level tag. If `bicycle=no` were a hard exclusion override, this way would be dropped from the cyclable network.

The same logic applies to `bicycle=use_sidepath`, which indicates that a compulsory parallel cycling facility exists nearby and the carriageway itself should not be used by cyclists. The following patterns illustrate how the inclusion-only approach handles different cases:

| Tagging pattern | Included? | Reasoning |
|---|---|---|
| `bicycle=no` + `cycleway=no` | No | No inclusion criterion met |
| `bicycle=no` + `cycleway=separate` | No | `cycleway=separate` is not an inclusion criterion |
| `cycleway=separate` + `bicycle=use_sidepath` | No | Neither tag is an inclusion criterion |
| `cycleway:right=separate` + `bicycle=use_sidepath` | No | Neither tag is an inclusion criterion |
| `bicycle=use_sidepath` alone | No | No inclusion criterion met |
| `cycleway:right=lane` + `bicycle=use_sidepath` | **Yes** | `cycleway:right=lane` is an inclusion criterion; a physical lane exists on one side |
| `bicycle:backward=use_sidepath` + `cycleway:right=lane` | **Yes** | `cycleway:right=lane` is an inclusion criterion; infrastructure is present |
| `highway=cycleway` + `bicycle=no` | **Yes** | `highway=cycleway` is an inclusion criterion; almost certainly a data quality issue |

In the cases where a way is included despite `bicycle=use_sidepath` or `bicycle=no` being present, the co-existence reflects genuine asymmetry: the carriageway has a cycle lane on one side and a compulsory sidepath on the other, or a restriction that applies to one direction or vehicle subtype rather than cycling as a whole. Including the way is the correct outcome because cycling infrastructure is physically present. If these tags were hard exclusion overrides, the way would be silently dropped and the infrastructure would disappear from the analysis. The inclusion-only approach surfaces the ambiguity rather than hiding it.

This conservative design means that for any way included in the cyclable network, at least one tag unambiguously signals cycling access or infrastructure. The approach does not guarantee that every included way is perfectly cyclable in all directions, asymmetries and data quality issues remain, but it does provide confidence that inclusion is grounded in a direct, positive signal rather than the absence of a prohibition. Full per-direction, per-side resolution of these asymmetries is performed for the bicycle route network analysis in stage 05 and is outside the scope of this derivation.

In [5]:
all_cyclable_ways_osm = duckdb.sql(f"""
SELECT * REPLACE(ST_GeomFromWKB(geometry) as geometry)
FROM '{osm_data_path}'
WHERE 1=1
AND osm_type = 'way'
AND map_contains(tags, 'highway')
AND (
    map_contains_entry(tags, 'bicycle', 'yes')
    OR map_contains_entry(tags, 'bicycle:forward', 'yes')
    OR map_contains_entry(tags, 'bicycle:backward', 'yes')

    OR map_contains_entry(tags, 'bicycle', 'designated')
    OR map_contains_entry(tags, 'bicycle:forward', 'designated')
    OR map_contains_entry(tags, 'bicycle:backward', 'designated')

    OR map_contains_entry(tags, 'bicycle', 'permissive')
    OR map_contains_entry(tags, 'bicycle:forward', 'permissive')
    OR map_contains_entry(tags, 'bicycle:backward', 'permissive')

    OR map_contains_entry(tags, 'bicycle', 'discouraged')
    OR map_contains_entry(tags, 'bicycle:forward', 'discouraged')
    OR map_contains_entry(tags, 'bicycle:backward', 'discouraged')

    OR map_contains_entry(tags, 'bicycle', 'optional_sidepath')
    OR map_contains_entry(tags, 'bicycle:forward', 'optional_sidepath')
    OR map_contains_entry(tags, 'bicycle:backward', 'optional_sidepath')

    OR map_contains_entry(tags, 'bicycle_road', 'yes')
    OR map_contains_entry(tags, 'cyclestreet', 'yes')

    OR map_contains_entry(tags, 'highway', 'cycleway')

    OR map_contains_entry(tags, 'cycleway', 'lane')
    OR map_contains_entry(tags, 'cycleway:both', 'lane')
    OR map_contains_entry(tags, 'cycleway:right', 'lane')
    OR map_contains_entry(tags, 'cycleway:left', 'lane')

    OR map_contains_entry(tags, 'cycleway', 'shared_lane')
    OR map_contains_entry(tags, 'cycleway:both', 'shared_lane')
    OR map_contains_entry(tags, 'cycleway:right', 'shared_lane')
    OR map_contains_entry(tags, 'cycleway:left', 'shared_lane')

    OR map_contains_entry(tags, 'cycleway', 'share_busway')
    OR map_contains_entry(tags, 'cycleway:both', 'share_busway')
    OR map_contains_entry(tags, 'cycleway:right', 'share_busway')
    OR map_contains_entry(tags, 'cycleway:left', 'share_busway')

    OR map_contains_entry(tags, 'cycleway', 'track')
    OR map_contains_entry(tags, 'cycleway:both', 'track')
    OR map_contains_entry(tags, 'cycleway:right', 'track')
    OR map_contains_entry(tags, 'cycleway:left', 'track')
)
""")

### 2.3 Clip to the Netherlands boundary

`all_cyclable_ways_osm` may include segments that extend beyond the Netherlands boundary. The cell below intersects each way with `nl_admin_area_gdf` (loaded via `01_boundaries`), drops ways falling entirely outside, and recomputes length in EPSG:28992. The rationale for the clip step is documented in `00_bicycle_route_relations`.

In [6]:
nl_admin_area_arrow = nl_admin_area_gdf.to_arrow()

all_cyclable_ways = duckdb.sql("""
SELECT w.* RENAME(w.length as length_osm_meters),
    ROUND(ST_Length(ST_Transform(ST_Intersection(nl.geometry, w.geometry), 'EPSG:4326', 'EPSG:28992', always_xy := true)), 2) AS length_nl_meters
FROM all_cyclable_ways_osm w
JOIN nl_admin_area_arrow nl
ON ST_Intersects(nl.geometry, w.geometry)
""")

### 2.4 Limitations

**Underestimation of the cyclable network.** The inclusion-only definition excludes ways where cycling is legally permitted by default but not signalled by any explicit tag. In the Netherlands, cycling is permitted on most roads unless explicitly prohibited. A `highway=residential` or `highway=tertiary` with no bicycle-related tags is legally cyclable, but will not appear in `all_cyclable_ways` under this approach. The magnitude of this underestimation is unknown; it depends on how many ways in the Dutch OSM extract carry neither explicit permission nor any infrastructure tag. As a result, `non_bicycle_route_ways` likely underestimates the full extent of ways a cyclist could legally use outside the bicycle route relations.

A more permissive definition, including all carriageway-type roads not explicitly prohibited, would reduce this underestimation but would also incorporate ways where cycling is merely tolerated rather than intended, making the variable harder to interpret as a measure of cycling provision.

The conservative approach is appropriate here because `non_bicycle_route_ways` is a secondary variable in this pipeline. The primary analytical focus, as established in the thesis Methodology (§3.3, §3.5), is `bicycle_route_m_ways_distinct`: the assessment of attribute completeness is confined to the designated bicycle route network, and cyclable ways outside any route relation are *"not judged complete or incomplete, but are not assessed"*. Underestimation in the non-route layer is therefore a known and accepted trade-off in exchange for a definition that is explicit, reproducible, and grounded directly in OSM tags rather than assumed defaults.

**Route relation membership as a partial remedy.** Ways that are members of bicycle route relations are insulated from this limitation, since the thesis treats route membership itself as evidence of cyclability. Inclusion in a route relation, even in the absence of explicit access or infrastructure tags, provides an additional signal that cycling is permitted: a mapper would not ordinarily include a way in a designated cycling route if cycling were prohibited or implausible. This signal is not infallible, but it means that `bicycle_route_m_ways_distinct` is likely more complete as a measure of cyclability than `non_bicycle_route_ways`, which relies on tags alone. The asymmetry is acceptable given the respective roles of the two variables in the analysis.

---

## 3. Exclude ways already part of a bicycle route network

Once `all_cyclable_ways` is defined, ways already captured in `bicycle_route_m_ways_distinct` are removed via an anti-join on `way_m_id`. The result is the set of ways that are cyclable but unaffiliated with any designated bicycle route relation.

In [8]:
non_bicycle_route_ways = duckdb.sql("""
SELECT *
FROM all_cyclable_ways w
WHERE NOT EXISTS (
    SELECT 1
    FROM bicycle_route_m_ways_distinct rw
    WHERE CAST(split_part(rw.way_m_id, '/', 1) AS BIGINT) = w.osm_id
)
""")

Because `bicycle_route_m_ways_distinct` is already deduplicated (one row per OSM way), and `all_cyclable_ways` is constructed directly from the OSM way table (also one row per way), no additional deduplication is needed here. The two sets are mutually exclusive by construction.

---

## 4. Result: `non_bicycle_route_ways`

`non_bicycle_route_ways` contains all ways that satisfy the cyclability definition above and are not members of any bicycle route relation. Each row is one OSM way. The cell below reports the total count and total clipped length.

In [13]:
total_ways = duckdb.sql("""
SELECT COUNT(*) AS ways_count
FROM non_bicycle_route_ways
""").fetchone()[0]

total_length_km = duckdb.sql("""
SELECT ROUND(SUM(length_nl_meters) / 1000, 3) AS total_km
FROM non_bicycle_route_ways
""").fetchone()[0]

print(f"Total cycleable ways outside bicycle route network: {total_ways:>10,}")
print(f"Total cycleable length outside bicycle route network: {total_length_km:>10,.3f} km")

Total cycleable ways outside bicycle route network:    249,920
Total cycleable length outside bicycle route network: 33,159.307 km


The table can be joined with spatial units (municipalities, provinces, H3 cells) in the downstream `06_non_bicycle_route_ways_per_spatial_unit` notebook, using the same spatial join logic applied to `bicycle_route_m_ways_distinct`. Per the thesis (§3.4.6) and the pipeline diagram, the resulting `non_bicycle_route_ways_per_spatial_unit` table feeds into `07_non_bicycle_route_extent_metrics`, which contributes to the MAUP robustness checks (stage 08) rather than to the headline classifiability results.